# Week 8b — Where is the information lost?

## Week 8 result
Fix B (graded, press-down expert) on the real connectome: **0/50** (random network 0/50). Carried to the zone 88% (random 96%), final distance 29 mm (random 36 mm). Outcomes: 26 still holding the cube, 17 put down in the zone but not finished, 7 outside the zone. A linear readout without a brain also scored 0/50, and the MLP 50/50, as planned.

**What the video shows:** one failure mode everywhere. Near the target the decoded commands shrink toward zero while the expert's don't (e.g. expert dy +10, dz −20 mm/step; decoded ≈ 0). So the arm stops wherever the commands cancel:
- **hovering** with the cube 30–35 mm from the target, about 9 cm up (as in week 7);
- or it **releases and never lifts away**: the grip opens correctly over the zone, but decoded dz stays near 0 where the expert says +20 mm/step up. The success test needs the gripper 6 cm clear.

**Why the sandbox missed it:** its stand-in brain decoded the expert's commands far better than the connectome does (dx R² 0.83 vs 0.27, dz 0.69 vs 0.39). It was calibrated to reproduce week 7's hovering, not week 7's decoding accuracy. From now on a stand-in is matched on decoding R² too.

**The diagnosis:** a ridge readout with little information predicts close to the average action, which is about zero motion. That is regression to the mean, and it limits any teacher. The question is **where the information is lost**:

| where | evidence so far | what would fix it |
|---|---|---|
| **at the input**: 21 JO neurons per channel since week 7 (week 5: 42) | encoder sweep R² fell from 0.76 (week 5) to 0.33–0.77 | more input neurons per channel, or higher rates |
| **inside the network** | week 5: JO spikes 0.95–0.98, all DNs 0.76–0.95 (4 signals) | a different input/output choice |
| **at the readout**: 1,332 DNs, only ~25% active | week 5: widening from 68 candidates to all DNs helped on every signal | read out from more of the brain |

## This notebook
1. **Record** (~10 min): the expert drives 40 episodes (graded expert, half with 5 mm noise) while the connectome watches. Recorded: the 335 JO input neurons, all 1,332 DNs, and every other neuron that fires ≥ 1 Hz during an expert episode (found first with one brain run).
2. **Compare** (~2 min, CPU): out-of-episode ridge, the same states for every readout:

| readout | why |
|---|---|
| JO input spikes | the ceiling: what the encoder delivers |
| all DNs | week 7/8's readout |
| 1,332 random active non-DNs | same size as the DNs: are the DNs special, or is it just the count? |
| 3,000 most informative active neurons (chosen inside each fold) | the affordable wide readout |
| all active neurons + DNs | the most the brain offers |
| 8 noise-free signals, no brain | reference |

Reported: expert dx, dy, dz R²; grip; **motion gain near the target** (1 = unbiased, near 0 = shrunk, the hover); **dz sign while lowering** and **while retreating**; the 8 signals.

3. **The rule (fixed before the run):** score = mean of dx, dy, dz R² and the two dz-sign agreements.
   - **3,000-neuron readout ≥ DNs + 0.10** → run the closed loop with it (Step 3, ~60 min): week 8's pipeline, fix B, `k = 3000` chosen on round 0, the same seeds. **Pass: ≥ 40/50.**
   - **JO input not ≥ DNs + 0.10 either** → the encoder is the bottleneck. Week 9 changes the input (more neurons per channel), not the readout.
   - **Otherwise** the information reaches the brain but not the readout we can afford.

Set `RUN_CLOSED_LOOP = True` to run Step 3 regardless, or `False` to stop after the comparison.

**A trade-off to note:** reading out from central neurons, not just the descending neurons, is still a frozen connectome plus a linear readout, so it remains "the brain did it". It is less like the real fly, though: in the fly only the DNs reach the body.

Needs: week 8's `sim/` (incl. `expert8.py`, `week8.py`), the new `sim/pool8.py`, and `data/week8/week8_connectome_b.pkl` (for the comparison table).

**Runs on Colab, Kaggle or the laptop.** On Kaggle: Accelerator **GPU T4 x2** (not P100), Internet **On**, inputs `arm-sim-code` and `arm-sim-data` (see `kaggle_migration.md`). Results go to `/kaggle/working/week8b/` and are saved with the version. To resume an unfinished run, attach that version's output as an input: its `week8b/` is copied in and `RESUME` carries on.

### Setup (Colab, Kaggle or laptop)

In [ ]:
import sys, subprocess, os, shutil, zipfile, re
from pathlib import Path

# ---- where are we? -------------------------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
KAGGLE_INPUT = Path('/kaggle/input')
IN_KAGGLE = not IN_COLAB and KAGGLE_INPUT.is_dir()
HOST = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'laptop'
if IN_COLAB or IN_KAGGLE:
    os.environ.setdefault('MUJOCO_GL', 'egl')                      # headless video; set before mujoco loads
    os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba_cache')   # the code folder is read-only on Kaggle

# ---- packages ------------------------------------------------------------------------------------
PKGS = ['brian2', 'numba', 'mujoco', 'imageio', 'imageio-ffmpeg']
if IN_COLAB or IN_KAGGLE:
    try:
        import brian2, numba, mujoco, imageio_ffmpeg
    except ImportError:
        if IN_KAGGLE:
            # A batch run (Save & Run All) can't survive a kernel restart, so numpy is pinned to the
            # installed version: nothing gets upgraded, nothing needs a restart.
            import numpy
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS, f'numpy=={numpy.__version__}'])
            if r.returncode:
                raise RuntimeError("pip install failed. Notebook settings (right panel) -> Internet: On. "
                                   "Kaggle only offers Internet to phone-verified accounts.")
            import brian2, numba, mujoco, imageio_ffmpeg
        else:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS], check=True)
            print("packages installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
            os.kill(os.getpid(), 9)

# ---- folders -------------------------------------------------------------------------------------
# Kaggle: the inputs are read-only and can come from several datasets and earlier notebook versions.
# They are linked into one tree with the Drive layout (arm_simulation/brain/{sim,data}, arm/,
# mujoco_menagerie/), so every path below is the same as on Colab. Results go to /kaggle/working.
def _link_tree(src, dst):
    """Link everything under src into dst, folder by folder, so several inputs can fill one tree."""
    dst.mkdir(parents=True, exist_ok=True)
    for s in sorted(src.iterdir()):
        d = dst / s.name
        if d.is_symlink() and s.is_dir() and d.is_dir():   # a folder another input already linked: split it
            t = d.resolve(); d.unlink(); _link_tree(t, d)
        if s.is_dir() and d.is_dir() and not d.is_symlink():
            _link_tree(s, d)
        elif not d.exists() and not d.is_symlink():
            d.symlink_to(s)

_RUN = re.compile(r'week\d+[a-z]*(_[a-z]+)?')        # week7, week8b, week8b_quick: a notebook's results folder

def _find_roots(top, depth=4):
    """('tree', p): p holds brain/, arm/ or mujoco_menagerie/ (an uploaded dataset).
       ('runs', p): p holds week*/ folders (an earlier notebook version's output)."""
    found = []
    def walk(p, d):
        try:
            kids = sorted(p.iterdir())
        except OSError:
            return
        names = {k.name for k in kids}
        zips = [z for z in kids if z.suffix == '.zip' and z.stem in ('brain', 'arm', 'mujoco_menagerie')
                and z.stem not in names]           # in case an upload kept its folders zipped
        if zips:
            out = Path('/tmp/kaggle_unzip') / p.name
            for z in zips:
                if not (out / z.stem).exists():
                    zf = zipfile.ZipFile(z)
                    inside = all(n.startswith(z.stem + '/') for n in zf.namelist())
                    zf.extractall(out if inside else out / z.stem)
            found.append(('tree', out))
        if d > 0 and names & {'brain', 'arm', 'mujoco_menagerie'}:
            found.append(('tree', p))
        elif d > 0 and any(k.is_dir() and _RUN.fullmatch(k.name) for k in kids):
            found.append(('runs', p))
        elif d < depth:
            for k in kids:
                if k.is_dir():
                    walk(k, d + 1)
    walk(top, 0)
    return found

if IN_KAGGLE:
    ROOT = Path('/tmp/arm_simulation')                     # links only, rebuilt every session
    shutil.rmtree(ROOT, ignore_errors=True)
    ROOTS = _find_roots(KAGGLE_INPUT)
    for kind, p in sorted(ROOTS, key=lambda r: r[0] == 'tree'):   # notebook outputs first: newest results win
        if kind == 'tree':
            _link_tree(p, ROOT)
        else:
            for k in sorted(p.iterdir()):
                if k.is_dir() and _RUN.fullmatch(k.name):
                    _link_tree(k, ROOT / 'brain' / 'data' / k.name)
    print("Kaggle inputs:", *[f"{k:5s} {p}" for k, p in ROOTS], sep="\n  ")
    BRAIN = ROOT / 'brain'
elif IN_COLAB:
    BRAIN = Path('/content/drive/MyDrive/arm_simulation/brain')
else:
    BRAIN = Path.home() / 'arm_simulation/brain'
DATA = BRAIN / 'data'
RUNS = Path('/kaggle/working') if IN_KAGGLE else DATA      # where this notebook's results go
ARM = next((p for p in [BRAIN.parent / 'arm', BRAIN.parent / 'week4/arm'] if (p / 'panda_env.py').exists()), None)
sys.path.insert(0, str(BRAIN))
if ARM: sys.path.insert(0, str(ARM))

def out_dir(name):
    """This notebook's results folder. On Kaggle it is /kaggle/working/<name>, saved with the version; if an
    earlier version's output is attached as an input, its <name>/ is copied in first, so RESUME carries on."""
    o = RUNS / name
    if RUNS != DATA and not o.exists() and (DATA / name).exists():
        shutil.copytree(DATA / name, o)
        print(f"resuming: copied the earlier run's {name}/ into {o}")
    o.mkdir(parents=True, exist_ok=True)
    return o

QUICK = False            # True: smoke test (8 recorded episodes; closed loop 3 x 6 DAgger, 10 test seeds), in week8b_quick/
RESUME = True
RUN_CLOSED_LOOP = 'auto' # 'auto': only if the rule says so; True: always; False: stop after the comparison
RUN_RANDOM = False       # after a connectome pass: the random network with its own wide readout (~70 min more)
K_WIDE = 3000            # wide readout size (ridge on 14,000 rows x 3,000 neurons: ~2-3 min per DAgger refit)
USE_GPU = True
GPU_DTYPE = 'float32'
MAX_BATCH = 64

GPU = None
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            GPU = torch.cuda.get_device_name(0)
            print(f"GPU: {GPU} (x{torch.cuda.device_count()}); torch {torch.__version__}; host {HOST}")
            if 'P100' in GPU:
                raise RuntimeError("Kaggle's current PyTorch has no kernels for the P100 (sm_60). "
                                   "Settings -> Accelerator -> GPU T4 x2.")
        else:
            print("USE_GPU = True but no GPU. Colab: Runtime -> Change runtime type -> T4 GPU. "
                  "Kaggle: Settings -> Accelerator -> GPU T4 x2.")
    except ImportError:
        print("torch is not installed here -> numba on the CPU")

for p in ['sim/batch7.py', 'sim/week7.py', 'sim/network7.py', 'sim/interface.py', 'sim/expert8.py',
          'sim/week8.py', 'sim/pool8.py', 'sim/video7.py', 'data/weights_male-cns_v1.0/meta.json',
          'data/shortlist_week1.csv', 'data/wsyn_choice.json', 'data/week8/week8_connectome_b.pkl',
          'data/week7/week7_random_network_meta.json']:
    print(f"{p:50s} {'ok' if (BRAIN / p).exists() else 'MISSING'}")
print(f"{'week 4 arm code':50s} {ARM if ARM else 'MISSING'}")
if ARM:
    print(f"{'mujoco_menagerie next to arm/':50s} "
          f"{'ok' if (ARM.resolve().parent / 'mujoco_menagerie/franka_emika_panda').exists() else 'MISSING'}")

### Load the connectome, the neuron groups and the arm (same as week 8)

In [ ]:
import json, time, gc, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore', module='brian2')
from sim.lif import WeightSet, peak_mem_gb
from sim.pickplace import SIGNALS8
from sim.interface import Encoder
from sim.batch7 import make_engine, BatchBrain
from sim.network7 import wiring_labels, random_network
from sim.week7 import reference_states, comparison_rows
from sim.expert8 import expert8
from sim.week8 import run_network8
from sim.pool8 import active_neurons, collect_expert, pool_report, gate, score
from panda_env import PandaPickPlace, OBS_NAMES

T_START = time.perf_counter()
W7, W8 = DATA / 'week7', DATA / 'week8'
OUT = out_dir('week8b_quick' if QUICK else 'week8b')

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
W_RAND = json.loads((W7 / 'week7_random_network_meta.json').read_text())['w_scale']
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
ALL_DN = ws.idx(ws.neurons['cls'].astype(str).str.contains('descending', case=False))
full = ws.subnetwork(np.arange(ws.N))
ws._out = None; gc.collect()
env = PandaPickPlace()

N_ROUNDS = 3 if QUICK else 4
N_PER_ROUND = 6 if QUICK else 30
CFG = dict(
    train_seeds=[list(range(64_000 + 1000 * r, 64_000 + 1000 * r + N_PER_ROUND)) for r in range(N_ROUNDS)],  # = weeks 7-8
    betas=[1.0] + [0.0] * (N_ROUNDS - 1),
    test_seeds=list(range(92_000, 92_000 + (10 if QUICK else 50))),                                         # = weeks 7-8
    cut_seeds=list(range(92_000, 92_000 + (3 if QUICK else 10))),
    k=None, n_video=2, use_gpu=GPU is not None, dtype=GPU_DTYPE, max_batch=MAX_BATCH, cpu_batch=1)
REC_SEEDS = list(range(96_000, 96_000 + (8 if QUICK else 40)))      # new seeds, used nowhere else
print(f"male CNS: {ws.N:,} neurons; JO-C/E {len(JO)}; DNs {len(ALL_DN)}; w_scale {W}; results in {OUT}")

## Step 1: which neurons fire, then record (about 10 min on a T4)

One brain run through a whole expert episode (reach → grasp → carry → lower → release → retreat) finds every non-input neuron firing ≥ 1 Hz. Then the expert drives 40 new episodes while the connectome watches, at week 8's timing (80 ms of brain per 40 ms step, 80 ms window), recording the JO inputs, all DNs and all active neurons.

In [ ]:
def wide_setup(sub, w_scale, label):
    """Encoder from this network's wiring (week 7's rule), its active neurons, an engine recording
    JO + DNs + active neurons."""
    keep = np.asarray(sub['keep'])
    enc = Encoder(JO, wiring_labels(sub, JO, 16), SIGNALS8, kind='wiring')
    eng = make_engine(sub, JO, GPU is not None, GPU_DTYPE)
    others = np.setdiff1d(np.arange(len(keep)), np.searchsorted(keep, JO))
    s_ep = reference_states(env, expert8, OBS_NAMES, n=60)
    t0 = time.perf_counter()
    act_local, hz = active_neurons(BatchBrain(eng, enc, ALL_DN), w_scale, s_ep, others)
    active = keep[act_local]
    print(f"{label}: {len(active):,} non-input neurons fire >= 1 Hz during an expert episode "
          f"({len(active) / len(others):.1%}); {np.isin(ALL_DN, active).sum()} of them are DNs  "
          f"[{time.perf_counter() - t0:.0f} s]")
    return enc, eng, active

ENC, ENG, ACTIVE = wide_setup(full, W, 'connectome')
RECORD = np.unique(np.r_[JO, ALL_DN, ACTIVE])
col = lambda glob: np.searchsorted(RECORD, np.asarray(glob))
D = collect_expert(BatchBrain(ENG, ENC, RECORD), W, OBS_NAMES, env, REC_SEEDS, expert8,
                   OUT / 'week8b_connectome_expert.npz', max_batch=MAX_BATCH, resume=RESUME)
del ENG; gc.collect()
if GPU:
    torch.cuda.empty_cache()
print(f"peak memory {peak_mem_gb():.2f} GB")

## Step 2: where the information is (deliverable 1)

In [ ]:
rng = np.random.default_rng(0)
NON_DN = np.setdiff1d(ACTIVE, ALL_DN)
WIDE = np.union1d(ACTIVE, ALL_DN)
POOLS = {
    'JO input spikes (ceiling)': col(JO),
    'all DNs (week 8)': col(ALL_DN),
    f'{len(ALL_DN):,} random active non-DNs': col(rng.choice(NON_DN, min(len(ALL_DN), len(NON_DN)), replace=False)),
    'active, 3,000 most informative': (col(WIDE), K_WIDE),
    f'all active + DNs': col(WIDE),
    '8 signals, noise-free (no brain)': ('signals',),
}
ROWS, COUNTS = pool_report(D, POOLS, expert8, OBS_NAMES)
REPORT = pd.DataFrame(ROWS).set_index('readout').drop(columns='s')
REPORT['score'] = [round(score(r), 3) for r in ROWS]
print(f"{COUNTS['rows']:,} steps from {COUNTS['episodes']} expert episodes; near the target {COUNTS['near_target']}, "
      f"lowering {COUNTS['lowering']}, retreating {COUNTS['retreat']}")
display(REPORT)
REPORT.to_csv(OUT / f'week8b_readouts_{HOST}.csv')
DECISION, WHY = gate(ROWS)
print(f"\nRULE -> {DECISION}: {WHY}")

## Step 3: closed loop with the wide readout (about 60 min; only if the rule says so)

Week 8's pipeline unchanged (fix B, 80/80 timing, DAgger 4 × 30, the same seeds and test), except the readout: the **3,000 most informative** of the active neurons + DNs, chosen once on round 0 (the expert-driven round), then a ridge readout on them. **Pass: ≥ 40/50.**

In [ ]:
RUN = (DECISION == 'closed loop') if RUN_CLOSED_LOOP == 'auto' else bool(RUN_CLOSED_LOOP)
NETS = {}
CONN8 = pickle.load(open(W8 / 'week8_connectome_b.pkl', 'rb')) if (W8 / 'week8_connectome_b.pkl').exists() else None
if RUN:
    CFGW = dict(CFG, k=K_WIDE)
    NETS['connectome, wide readout'] = run_network8('connectome, wide readout', full, JO, WIDE, env, OBS_NAMES, CFGW,
                                                    OUT, w_scale=W, fix='B', resume=RESUME, extra_record=DN_cand)
    s = NETS['connectome, wide readout']['summary']
    PASS = s['success'] >= 0.8 * s['n']                    # 40/50
    print(f"\nCONNECTOME, WIDE READOUT: success {s['success']}/{s['n']} (95% CI {s['ci'][0]:.0%}-{s['ci'][1]:.0%}), "
          f"placement {s['place_mm_success']} mm, time {s['time_s']} s, {s['outcomes']}")
    print(f"PASS (>= 80% of test seeds, 40/50): {'YES' if PASS else 'NO'}")
    print("while collecting, by round:", [f"{r['success']:.0%}" for r in NETS['connectome, wide readout']['dagger']['log']])
    if PASS and RUN_RANDOM:
        rnet = random_network(full, seed=0)
        _, eng_r, ACT_R = wide_setup(rnet, W_RAND, 'random network')
        del eng_r; gc.collect()
        NETS['random network, wide readout'] = run_network8('random network, wide readout', rnet, JO,
                                                            np.union1d(ACT_R, ALL_DN), env, OBS_NAMES, CFGW, OUT,
                                                            w_scale=W_RAND, fix='B', resume=RESUME, extra_record=DN_cand)
        del rnet; gc.collect()
else:
    PASS = None
    print("closed loop not run:", WHY)

## Step 4: comparison, video, results file

In [ ]:
SUMM = {k: v['summary'] for k, v in NETS.items()}
if CONN8 is not None:
    SUMM['connectome, DN readout (week 8)'] = CONN8['summary']
if SUMM:
    TABLE = pd.DataFrame(comparison_rows(SUMM)).set_index('policy')
    display(TABLE.style.format({c: '{:.0%}' for c in ('success rate', 'lifted', 'carried to zone', 'stalled')}, na_rep='-'))
    display(pd.DataFrame({k: s['outcomes'] for k, s in SUMM.items()}).fillna(0).astype(int).T)
    if len(NETS) == 2:
        from sim.week7 import verdict
        V, _ = verdict(NETS['connectome, wide readout']['outs'], NETS['random network, wide readout']['outs'])
        print("VERDICT (wide readouts):", V)
VIDEO = None
if NETS:
    EPISODES = [e for k in NETS for e in NETS[k]['video_eps']]
    for e in EPISODES:
        print(f"{e['net']}, seed {e['seed']}: {e['outcome']}")
    pickle.dump(EPISODES, open(OUT / 'week8b_episodes.pkl', 'wb'))
    from sim.video7 import render
    VIDEO = OUT / f'week8b_pick_place_{HOST}.mp4'
    if render(str(OUT / 'week8b_episodes.pkl'), str(VIDEO), slow=0.5):
        from IPython.display import Video
        display(Video(str(VIDEO), embed=True, width=900))
summary = dict(host=HOST, quick=QUICK, gpu=GPU, n_active=int(len(ACTIVE)), n_record=int(len(RECORD)),
               counts=COUNTS, readouts=REPORT.reset_index().to_dict('records'), decision=DECISION, why=WHY,
               closed_loop=bool(RUN), passed=PASS, k_wide=K_WIDE,
               table=TABLE.reset_index().to_dict('records') if SUMM else None,
               outcomes={k: s['outcomes'] for k, s in SUMM.items()},
               networks={k: dict(dagger_log=v['dagger']['log'], cut=v['cut']['summary'] if 'cut' in v else None,
                                 wall_min=v['wall_min']) for k, v in NETS.items()},
               wall_min=(time.perf_counter() - T_START) / 60)
(OUT / f'week8b_results_{HOST}.json').write_text(json.dumps(summary, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)))
print(f"this session {summary['wall_min']:.0f} min. Send me week8b_results_{HOST}.json"
      f"{' and the video' if VIDEO else ''} from {OUT}.")